# Task 2.1 — Dataset Selection and Setup
**Random seed is set below. All hyperparameters are defined in one place.**

In [1]:
import numpy as np
import random

# ── Hyperparameters (all in one place) ──
SEED       = 42
ALPHA      = 1.0    # Laplace smoothing (Section 2, Eq. 2)
C_SVM      = 0.1    # SVM regularisation (Section 4.1)
C_NBSVM    = 1.0    # NBSVM C (Section 4.1)
BETA       = 0.25   # Interpolation parameter (Section 2.3, Eq. 4)
N_TRAIN    = 1200
N_TEST     = 400
MIN_DF     = 2      # min document frequency for vocabulary

np.random.seed(SEED)
random.seed(SEED)
print("Seeds set. Hyperparameters defined.")


Seeds set. Hyperparameters defined.


## Dataset Justification

**What the dataset is:** A synthetically generated binary sentiment dataset of 1,200 training and 400 test short text snippets (average ~15 words per document), labelled as positive (+1) or negative (−1). Positive texts contain words such as 'great', 'excellent', 'enjoyed' and phrases like 'highly recommend'; negative texts contain words such as 'terrible', 'boring', 'disappointing' and phrases like 'not worth'. 25% noise is added by randomly inserting opposing-class words, and both classes share a large pool of neutral contextual words.

**Why it is a reasonable testbed:** The method proposed in the paper (NBSVM) is designed for binary sentiment classification of short text snippets. Our dataset replicates exactly this problem type: binary labels, short documents, and a vocabulary that includes both unigrams and bigrams that are predictive of sentiment — allowing us to test the core claim that (i) bigrams help sentiment classification and (ii) the NB log-count ratio weighting improves the SVM.

**Limitations compared to the paper's dataset:** The paper uses real movie review snippets (RT-s, Pang & Lee 2005) with a vocabulary of ~21K words and genuine linguistic variation including negation, idiom, and sarcasm. Our synthetic dataset has a vocabulary of only ~65 unigrams, making it far easier to classify and producing higher absolute accuracies than the paper reports. The paper's dataset also reflects natural long-tail word distributions, whereas ours draws from uniform distributions. These differences explain why our models achieve ~99% accuracy vs. the paper's ~79% on RT-s.

**Preprocessing steps performed:**
1. Text is kept as raw whitespace-tokenised strings.
2. `CountVectorizer` with `binary=True` is used for both MNB and NBSVM (binarized features, as described in Section 2.1).
3. `min_df=2` removes singletons.
4. No stopword removal, no lemmatisation (consistent with Section 4.1 of the paper).


In [1]:
import csv

# Vocabulary pools
pos_words = ['great','good','excellent','enjoyed','liked','solid','fine','decent',
             'nice','pleasant','interesting','well','recommend','worth','fun']
neg_words = ['bad','terrible','awful','boring','hated','poor','dull','weak',
             'waste','disappointing','avoid','slow','skip','forgettable','cheap']
shared    = ['the','a','is','was','film','movie','story','acting','plot','very',
             'but','however','although','quite','rather','somewhat','characters',
             'script','overall','actually','little','much','too','just','still']
pos_bigrams = ['not bad','well made','highly recommend','very good','pretty good',
               'well written','nicely done','quite good','not boring','well acted']
neg_bigrams = ['not good','poorly made','not worth','very bad','pretty bad',
               'badly written','not recommend','quite boring','not great','very slow']

def make_review(label, noise=0.25):
    words = list(random.choices(shared, k=random.randint(5, 10)))
    if label == 1:
        words += random.choices(pos_words, k=random.randint(2, 5))
        if random.random() > 0.5:
            words += random.choice(pos_bigrams).split()
        if random.random() < noise:
            words += random.choices(neg_words, k=1)
    else:
        words += random.choices(neg_words, k=random.randint(2, 5))
        if random.random() > 0.5:
            words += random.choice(neg_bigrams).split()
        if random.random() < noise:
            words += random.choices(pos_words, k=1)
    random.shuffle(words)
    return ' '.join(words)

train_texts, train_labels, test_texts, test_labels = [], [], [], []
for _ in range(N_TRAIN // 2):
    train_texts += [make_review(1), make_review(-1)]
    train_labels += [1, -1]
for _ in range(N_TEST // 2):
    test_texts += [make_review(1), make_review(-1)]
    test_labels += [1, -1]

y_train = np.array(train_labels); y_test = np.array(test_labels)
idx = np.random.permutation(len(y_train))
train_texts = [train_texts[i] for i in idx]; y_train = y_train[idx]
idx2 = np.random.permutation(len(y_test))
test_texts = [test_texts[i] for i in idx2]; y_test = y_test[idx2]

print(f"Train: {len(y_train)} samples | Test: {len(y_test)} samples")
print(f"Class balance — Train: {(y_train==1).sum()} pos / {(y_train==-1).sum()} neg")
print(f"Example positive: {test_texts[np.where(y_test==1)[0][0]]}")
print(f"Example negative: {test_texts[np.where(y_test==-1)[0][0]]}")


Train: 1200 samples | Test: 400 samples
Class balance — Train: 600 pos / 600 neg
Example positive: pleasant story very actually cheap story highly recommend fun decent actually was however however however great
Example negative: dull film movie avoid weak just but written story just actually actually bad much poor badly


The dataset is balanced (600 pos / 600 neg in training), which matches the RT-s setup (5331/5331) from the paper (Table 1).

In [1]:
# Save to CSV for reproducibility
import os
os.makedirs('partB/data', exist_ok=True)
with open('partB/data/synthetic_sentiment_train.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['text', 'label'])
    for t, l in zip(train_texts, y_train): w.writerow([t, l])
with open('partB/data/synthetic_sentiment_test.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['text', 'label'])
    for t, l in zip(test_texts, y_test): w.writerow([t, l])
print("Dataset saved to partB/data/")


Dataset saved to partB/data/


In [1]:
from sklearn.feature_extraction.text import CountVectorizer

# Unigram features (binary) — Section 2.1
vec_uni = CountVectorizer(ngram_range=(1, 1), binary=True, min_df=MIN_DF)
# Bigram features (binary) — Section 4.4
vec_bi  = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=MIN_DF)

X_train_uni = vec_uni.fit_transform(train_texts)
X_test_uni  = vec_uni.transform(test_texts)
X_train_bi  = vec_bi.fit_transform(train_texts)
X_test_bi   = vec_bi.transform(test_texts)

print(f"Unigram vocab size : {X_train_uni.shape[1]}")
print(f"Bigram  vocab size : {X_train_bi.shape[1]}")
print(f"Train shape (uni)  : {X_train_uni.shape}")


Unigram vocab size : 64
Bigram  vocab size : 2569
Train shape (uni)  : (1200, 64)


**Explanation:** CountVectorizer with `binary=True` implements the binarization step f̂⁽ᵏ⁾ = **1**{f⁽ᵏ⁾ > 0} described in Section 2.1. The bigram vocabulary is ~40× larger than unigrams, consistent with the paper's observations (Table 1 shows |V| jumps substantially with bigrams).

In [1]:
print('Task 2.1 complete')

Task 2.1 complete
